# Spin-up Quick Check Notebook
This notebook is organized for spin-up diagnostics on monthly model output.

## Task 1: Domain map animation of a selected biological variable
- Input file: `dws_500m.3d.201501.nc`
- Goal: animate a selected variable over time on the model map
- Output: an animated GIF (optional) and inline animation preview

## Task 2: Aggregated time series of a selected biological variable for the whole domain
- Cell 2 (Task 2): build and plot the domain-mean trend at a selected layer (4d) or 3d variable
- Input file: `dws_500m.3d.201501.nc`
- Goal: visualize time series of spatially averaged values over the whole domain also for a specific vertical level;
- Output: a time-series line plot

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

from scipy.interpolate import RegularGridInterpolator
from matplotlib.animation import FuncAnimation
from pathlib import Path
from IPython.display import HTML

import pandas as pd

In [ ]:
# meta data:
#! BFM biological model

# pelagic variables (group PelVariables):
#! pelagic  (O)              O2:   Oxygen (mmol/m3)
#! pelagic  (P)              N1:   Phosphate (mmol/m3)
#! pelagic  (N)              N3:   Nitrate (mmol/m3)
#! pelagic  (N)              N4:   Ammonium (mmol/m3)
#! pelagic  (Si)             N5:   Silicate (mmol/m3)
#! pelagic  (R)              N6:   Reduction Equivalents (mmol/m3)
#! pelagic  (N)              O4:   N2-sink (mmol/m3)
#! pelagic  (CNP)            B1:   Pelagic Bacteria
#! pelagic  (CNPSiI)         P1:   Diatoms (group PhytoPlankton))
#! pelagic  (CNPSiI)         P2:   Flagellates (group PhytoPlankton))
#! pelagic  (CNPSiI)         P3:   PicoPhytoPlankton (group PhytoPlankton))
#! pelagic  (CNPSiI)         P4:   Dinoflagellates (group PhytoPlankton))
#! pelagic  (CNP)            Z3:   Carnivorous mesozooplankton (group MesoZooPlankton))
#! pelagic  (CNP)            Z4:   Omnivorous mesozooplankton (group MesoZooPlankton))
#! pelagic  (CNP)            Z5:   Microzooplankton (group MicroZooPlankton))
#! pelagic  (CNP)            Z6:   Heterotrophic nanoflagellates (HNAN) (group MicroZooPlankton))
#! pelagic  (CNPSi)          R1:   Labile Organic Carbon (LOC)
#! pelagic  (C)              R2:   CarboHydrates (sugars)
#! pelagic  (CNPSi)          R6:   Particulate Organic Carbon (POC)
#! pelagic  (C)              R7:   Refractory Disoolved Organic Carbon

# Benthic variables (group BenVariables):
# ! benthic  (CNP)            Y1:   Epibenthos (group BenOrganisms))
# ! benthic  (CNP)            Y2:   Deposit feeders (group BenOrganisms))
# ! benthic  (CNP)            Y3:   Suspension feeders (group BenOrganisms))
# ! benthic  (CNP)            Y4:   Meiobenthos (group ! BenOrganisms))
# ! benthic  (CNP)            Y5:   Benthic predators (group BenOrganisms))
# ! benthic  (CNPSi)          Q1:   Labile organic carbon (group BenDetritus))
# ! benthic  (CNPSi)          Q11:  Labile organic carbon (group BenDetritus))
# ! benthic  (CNPSi)          Q6:   Particulate organic carbon (group BenDetritus))
# ! benthic  (CNP)            H1:   Aerobic benthic bacteria (group BenBacteria))
# ! benthic  (CNP)            H2:   Anaerobic benthic bacteria (group BenBacteria))
# ! benthic  (P)              K1:   Phosphate in oxic layer (group BenthicPhosphate))
# ! benthic  (P)              K11:  Phosphate in denit layer (group BenthicPhosphate))
# ! benthic  (P)              K21:  Phosphate in anoxic layer (group BenthicPhosphate))
# ! benthic  (N)              K4:   Ammonium in oxic layer (group BenthicAmmonium))
# ! benthic  (N)              K14:  Ammonium in denit layer (group BenthicAmmonium))
# ! benthic  (N)              K24:  Ammonium in anoxic layer (group BenthicAmmonium))
# ! benthic  (R)              K6:   Reduction equivalents 
# ! benthic  (M)              D1:   Oxygen penetration depth
# ! benthic  (M)              D2:   Denitrification depth 
# ! benthic  (M)              D6:   Depth distribution factor organic C 
# ! benthic  (M)              D7:   Depth distribution factor organic N
# ! benthic  (M)              D8:   Depth distribution factor organic P
# ! benthic  (M)              D9:   Depth distribution factor organic Si
# ! benthic  (O)              G2:   Benthic O2



# Jetty dataset meta data:
# TSM: mg/L
# C: mg/m3
# TOC: mgC/L
# POC: mgC/L
# DOC: mgC/L

In [ ]:
# setup
# Time series trends for all variables in vars_list over the full year 2015
DATA_DIR = Path('/export/lv9/projects/dws/model_output/archived_runs/')
FILE_PATTERN = 'dws_500m.3d.2015??.nc'
SURFACE_LAYER_INDEX = 10   # Use top 11; surface layer by default
USE_DAILY_MEAN = False     # Set True if you want daily-mean smoothing

Validation_DATA_DIR = Path('/export/lv9/projects/dws/results/validation/pelagic/')
Marsdiep_ts = 'Jetty_ts.csv'
CHLA_VAR = 'Chla'
ELEV_VAR = 'elev'
Bathymetry_VAR = 'bathymetry'  # Preferred name for bathymetry variable; will try alternatives if not found.

Benthic_POC_ts = '20100215_PAM_overview_1974_2009i.xlsx'

# Subdomain index range (Python slice: start inclusive, stop exclusive).
# Salt marsh zone nearby Miedema (unusual discontinuity in derived total Chla)
#X_SLICE = (215, 225)
#Y_SLICE = (133, 136)

# Marsdiep zone
#X_SLICE = (75, 78)
# Y_SLICE = (95, 98)

# Lauwesoog zone
X_SLICE = (245, 280)
Y_SLICE = (135, 150)

# Chla layer index to inspect and neighboring layers.
CHLA_LAYER_INDEX = 5  # top=11, bottom=1 in your convention

ROLLING_WINDOW = None  # e.g., 3 for smoothing, or None

#PP_csv_path = Validation_DATA_DIR / Marsdiep_PP_ts

# Parameters for model-observation comparison
MODEL_SURFACE_LAYER_INDEX = 10  # top=11, bottom=1 in your convention

# The time series data for Marsdiep is available from the NIOZ Dataverse at http://doi.org/10.25850/nioz/7b.b.5j

In [ ]:
# Helpers
# Reuse opened dataset if available; otherwise open yearly files.

def _find_time_dim(da: xr.DataArray) -> str:
    for d in da.dims:
        if 'time' in d.lower():
            return d
    raise ValueError(f'No time dimension found in {da.dims}')

def _find_vertical_dim(da: xr.DataArray, time_dim: str) -> str | None:
    candidates = ('level', 'z', 'sigma', 'layer', 'lev', 'depth', 'nmesh2_layer_3d')
    for d in da.dims:
        if d != time_dim and any(k in d.lower() for k in candidates):
            return d
    return None
   
def _drop_duplicate_time(da: xr.DataArray, time_dim: str) -> xr.DataArray:
    # Keep first occurrence when duplicate time stamps are present.
    time_values = np.asarray(da[time_dim].values)
    _, keep_idx = np.unique(time_values, return_index=True)
    keep_idx = np.sort(keep_idx)
    if keep_idx.size < time_values.size:
        da = da.isel({time_dim: keep_idx})
    return da
    
def _find_bathy_name(ds: xr.Dataset, preferred: str) -> str:
    if preferred in ds.variables:
        return preferred
    candidates = ('bathymetry', 'depth', 'h', 'H', 'bathy', 'bat', 'topo', 'd', 'water_depth')
    for name in candidates:
        if name in ds.variables:
            return name
    raise KeyError(
        f"Bathymetry variable not found. Tried '{preferred}' and {candidates}. "
        f"Available vars include: {list(ds.variables)[:30]}"
    )

def _to_float(values) -> np.ndarray:
    if np.ma.isMaskedArray(values):
        values = np.ma.filled(values, np.nan)
    return np.asarray(values, dtype=float)

def _pick_coord_name(ds: xr.Dataset, candidates: tuple[str, ...]) -> str | None:
    for name in candidates:
        if name in ds.variables:
            return name
    return None

def _maybe_smooth(series: xr.DataArray, time_dim: str) -> xr.DataArray:
    out = series
    if USE_DAILY_MEAN:
        out = out.resample({time_dim: '1D'}).mean(skipna=True)
    if ROLLING_WINDOW is not None:
        if ROLLING_WINDOW < 1:
            raise ValueError('ROLLING_WINDOW must be >= 1 or None')
        out = out.rolling({time_dim: ROLLING_WINDOW}, center=True).mean()
    return out

In [ ]:
# List of variables for analysis and visualization
vars_list = [
    'elev',
    #'temp',
    #'salt',
    #'O2o', 
    'netPPm2',
    #'N1p',
    #'N3n',
    #'N4n',
    #'N5s',
    #'N6r',
#         'B1c',
#         'Bac',
          'P1c',
#     'P2c',
#     'P3c',
#         'P4c',
#         'P5c',
#         'P6c',
#	      'P1l',
#         'P2l',
#         'P3l',
#         'P4l',
#         'P5l',
#         'P6l',
#         'Z2c',
#         'Z3c',
#         'Z4c',
#         'Z5c',
#         'Z6c',
#         'R1c',
#         'R2c',
#         'R3c',
#          'R6c',
#         'RZc',
#         'Q1c',
#         'Q11c',
#          'Q6c',
          'Chla',
#          'H1c',
#          'H2c',
#          'HNc',
#          'Hac',
          'Y1c',
          'Y2c',
          'Y3c',
#          'Y4c',
          'Y5c',
#          'Yy3c',
#          'K6r',
#          'K16r',
#          'K26r',
#          'K5s',
#          'K15s',
#          'K3n',
#          'K4n',
#         'K13n',
#          'K14n',
#          'K24n',
#          'K1p',
#          'K11p',
#         'K21p',
#          'D1m',
#          'D2m',
#          'O3c',
#          'pCO2',
#          'CO2',
#          'HCO3',
#          'CO3',
#          'pH',
#          'Ac'
#          'G3h'
#          'G13h'
#          'G23h'
#          'G3c'
#          'G13c'
#          'G23c'
#          'G14n'
#          'Acae'
#          'Acan'
#          'DICae'
#          'DICan'
#          'pHae'
#          'pHan'
#          'pCO2ae'
#          'pCO2an'
#          'G3h'
#          'G13h'
#          'BP1c'
#          'ETW',
#          'ESS'
#          'irrenh',
#          'turenh',
'xEPS'      
]

In [ ]:
spinup_datasets = {}

for i in range(1, 9):
    Spinup_DIR = DATA_DIR / f"spinup_0{i}"
    files = sorted(Spinup_DIR.glob(FILE_PATTERN))

    if not files:
        raise FileNotFoundError(f'No files found with pattern: {FILE_PATTERN} in {Spinup_DIR}')

    ds = xr.open_mfdataset(
        files,
        combine='nested',
        concat_dim='time',
        decode_times=True,
        data_vars='minimal',
        coords='minimal',
        compat='override',
        join='override',
    )

    spinup_datasets[f"spinup_{i:02d}"] = ds


In [ ]:
# Goal: show the selected subdomain on a bathymetry map.
bathy_name = _find_bathy_name(ds, Bathymetry_VAR)
bathy = ds[bathy_name].squeeze(drop=True)

# Find horizontal dims from bathymetry; if a time dim exists, use first timestep.
bathy_dims = list(bathy.dims)
time_like = [d for d in bathy_dims if 'time' in d.lower()]
if time_like:
    bathy = bathy.isel({time_like[0]: 0})
    bathy_dims = [d for d in bathy.dims if d != time_like[0]]

if len(bathy_dims) != 2:
    raise ValueError(f'Expected 2D bathymetry after squeezing, got dims {bathy.dims}')
y_dim, x_dim = bathy_dims

ny = bathy.sizes[y_dim]
nx = bathy.sizes[x_dim]
y0, y1 = Y_SLICE
x0, x1 = X_SLICE
if not (0 <= y0 < y1 <= ny):
    raise IndexError(f'Y_SLICE={Y_SLICE} out of bounds for {y_dim} size {ny}')
if not (0 <= x0 < x1 <= nx):
    raise IndexError(f'X_SLICE={X_SLICE} out of bounds for {x_dim} size {nx}')

lon_name = _pick_coord_name(ds, ('lonc', 'lon', 'longitude'))
lat_name = _pick_coord_name(ds, ('latc', 'lat', 'latitude'))

use_geo = False
if lon_name is not None and lat_name is not None:
    lon_da = ds[lon_name]
    lat_da = ds[lat_name]
    if y_dim in lon_da.dims and x_dim in lon_da.dims and y_dim in lat_da.dims and x_dim in lat_da.dims:
        x_plot = _to_float(lon_da.transpose(y_dim, x_dim).values)
        y_plot = _to_float(lat_da.transpose(y_dim, x_dim).values)
        if x_plot.shape == (ny, nx) and y_plot.shape == (ny, nx):
            if np.isfinite(x_plot).all() and np.isfinite(y_plot).all():
                use_geo = True

if not use_geo:
    x_plot, y_plot = np.meshgrid(np.arange(nx, dtype=float), np.arange(ny, dtype=float))

bathy2d = _to_float(bathy.transpose(y_dim, x_dim).values)

fig, ax = plt.subplots(figsize=(8.6, 6.8))
mesh = ax.pcolormesh(
    x_plot,
    y_plot,
    np.ma.masked_invalid(bathy2d),
    shading='auto',
    cmap='cividis',
)
cbar = fig.colorbar(mesh, ax=ax, fraction=0.04, pad=0.03)
units = bathy.attrs.get('units', '')
cbar.set_label(f'{bathy_name} [{units}]' if units else bathy_name)

# Draw the selected subdomain as a closed polygon on the map.
if use_geo:
    x_poly = [x_plot[y0, x0], x_plot[y0, x1 - 1], x_plot[y1 - 1, x1 - 1], x_plot[y1 - 1, x0], x_plot[y0, x0]]
    y_poly = [y_plot[y0, x0], y_plot[y0, x1 - 1], y_plot[y1 - 1, x1 - 1], y_plot[y1 - 1, x0], y_plot[y0, x0]]
    ax.plot(x_poly, y_poly, color='red', lw=2.2, label='Selected subdomain')
else:
    x_poly = [x0, x1, x1, x0, x0]
    y_poly = [y0, y0, y1, y1, y0]
    ax.plot(x_poly, y_poly, color='red', lw=2.2, label='Selected subdomain')

ax.set_title('Subdomain location on bathymetry map')
ax.set_xlabel('Longitude' if use_geo else x_dim)
ax.set_ylabel('Latitude' if use_geo else y_dim)
ax.grid(True, alpha=0.2)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

print(f'Bathymetry variable used: {bathy_name}')
print(f'Subdomain used: y={Y_SLICE[0]}:{Y_SLICE[1]}, x={X_SLICE[0]}:{X_SLICE[1]}')

In [ ]:
fig, axes = plt.subplots(
    len(vars_list),
    1,
    figsize=(14, 3 * len(vars_list)),
    sharex=True,
    constrained_layout=True,
)

if len(vars_list) == 1:
    axes = [axes]

failed_vars = []
missing_vars = []

for ax, vname in zip(axes, vars_list):

    all_values = []
    boundaries = [0]
    spin_labels = []
    units = ""

    for spin_name in sorted(spinup_datasets.keys()):

        ds = spinup_datasets[spin_name]

        try:

            if vname not in ds.variables:
                missing_vars.append((spin_name, vname))
                continue

            da = ds[vname].squeeze(drop=True)

            time_dim = _find_time_dim(da)
            z_dim = _find_vertical_dim(da, time_dim)

            # Select surface layer if present
            if z_dim is not None:
                da = da.isel({z_dim: SURFACE_LAYER_INDEX})

            # Remove duplicate timestamps
            da = _drop_duplicate_time(da, time_dim)

            # Spatial average
            spatial_dims = [d for d in da.dims if d != time_dim]

            if len(spatial_dims) == 0:
                raise ValueError("No spatial dimensions available.")

            # Variable-specific masking
            if vname.lower() in ("chla", "netppm2"):
                da = da.where(da >= 0)

            if vname.lower() == "etw":
                da = da.where(da >= -10)

            if vname.lower() == "xEPS":
                da = da.where(da >= 0)

            series = da.mean(dim=spatial_dims, skipna=True)

            # Daily mean / rolling mean
            series = _maybe_smooth(series, time_dim)

            if units == "":
                units = da.attrs.get("units", "")

            values = series.values

            all_values.append(values)

            boundaries.append(boundaries[-1] + len(values))
            spin_labels.append(spin_name)

        except Exception as e:
            failed_vars.append((spin_name, vname, str(e)))

    # ----------------------------------------------------
    # Concatenate all spinups into one long time series
    # ----------------------------------------------------
    long_series = np.concatenate(all_values)
    x = np.arange(len(long_series))

    ax.plot(x, long_series, lw=1.4, color="tab:blue")

    # Draw spinup boundaries
    ymax = np.nanmax(long_series)
    ymin = np.nanmin(long_series)

    for i, b in enumerate(boundaries[:-1]):

        ax.axvline(b, color="gray", ls="--", lw=0.8, alpha=0.6)

        # Put spinup label in the middle of each segment
        if i < len(spin_labels):
            center = (boundaries[i] + boundaries[i+1]) / 2
            ax.text(
                center,
                ymax,
                spin_labels[i],
                ha="center",
                va="bottom",
                fontsize=8,
            )

    ylabel = f"{vname} [{units}]" if units else vname

    ax.set_ylabel(ylabel)
    ax.set_title(vname)
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Model timestep (concatenated spinups)")

plt.show()

In [ ]:
csv_path = Validation_DATA_DIR / Marsdiep_ts

if not csv_path.exists():
    raise FileNotFoundError(f'CSV file not found: {csv_path}')

measurement_df = pd.read_csv(
    csv_path,
    na_values=['NA', ''],
    parse_dates=['timestamp'],
)

print(f'Loaded CSV: {csv_path}')
print(f'Shape: {measurement_df.shape[0]} rows x {measurement_df.shape[1]} columns')
print('Columns:')
print(list(measurement_df.columns))
print('')
print('First 5 rows:')
display(measurement_df.head())



In [ ]:
# Goal: Compare model results with observations for temporal dynamics of Chla and other variables nearby Marsdiep.


#Obs_VAR_NAME = 'TSM'
#Model_VAR_NAME = 'ESS'

#Obs_VAR_NAME = 'Daily_PP'
#Model_VAR_NAME = 'netPPm2'

#Obs_VAR_NAME = 'POC'
#Model_VAR_NAME = 'R6c'

Obs_VAR_NAME = 'NH4'
Model_VAR_NAME = 'N4n'

if Model_VAR_NAME not in ds.variables:
    raise KeyError(f"Variable '{Model_VAR_NAME}' not found. Available: {sorted(ds.data_vars)}")

Model_ds = ds[Model_VAR_NAME].squeeze(drop=True)
time_dim = _find_time_dim(Model_ds)

z_dim = _find_vertical_dim(Model_ds, time_dim)   # <-- may be None for 2D variables

# ---------------------------------------------------------
# 1. Handle 2D vs 3D variables
# ---------------------------------------------------------
if z_dim is None:
    # 2D variable: dims = (time, y, x)
    spatial_dims = [d for d in Model_ds.dims if d != time_dim]
    if len(spatial_dims) != 2:
        raise ValueError(f"Expected 2D variable with dims (time,y,x), got {Model_ds.dims}")
    y_dim, x_dim = spatial_dims
    Model_sub = Model_ds.isel({y_dim: slice(Y_SLICE[0], Y_SLICE[1]),
                               x_dim: slice(X_SLICE[0], X_SLICE[1])})
else:
    # 3D variable: dims = (time, z, y, x)
    xy_dims = [d for d in Model_ds.dims if d not in (time_dim, z_dim)]
    if len(xy_dims) != 2:
        raise ValueError(f'Expected 2 horizontal dims for {Model_VAR_NAME}, got {xy_dims}')
    y_dim, x_dim = xy_dims

    Model_sub = Model_ds.isel({
        z_dim: MODEL_SURFACE_LAYER_INDEX,
        y_dim: slice(Y_SLICE[0], Y_SLICE[1]),
        x_dim: slice(X_SLICE[0], X_SLICE[1])
    })

# ---------------------------------------------------------
# 2. Compute spatial mean
# ---------------------------------------------------------
model_series = Model_sub.mean(dim=(y_dim, x_dim), skipna=True)
model_series = _drop_duplicate_time(model_series, time_dim)
model_series = _maybe_smooth(model_series, time_dim)
model_series = model_series.where(model_series >= 0)

# Day of year
model_series['doy'] = model_series[time_dim].dt.dayofyear

# ---------------------------------------------------------
# 3. Observations
# ---------------------------------------------------------
if Obs_VAR_NAME not in measurement_df.columns:
    raise KeyError(f"Column {Obs_VAR_NAME} not found in the file.")

obs_df = measurement_df[['timestamp', Obs_VAR_NAME]].dropna(subset=['timestamp', Obs_VAR_NAME]).copy()
obs_df['timestamp'] = pd.to_datetime(obs_df['timestamp'], dayfirst=True, errors='coerce')
obs_df = obs_df.sort_values('timestamp')

if obs_df.empty:
    raise ValueError('No measurements found.')

obs_df['doy'] = obs_df['timestamp'].dt.dayofyear

# ---------------------------------------------------------
# 4. Plot
# ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 4.8))

ax.scatter(
    model_series['doy'],
    model_series.values,
    s=18,
    color='tab:blue',
    label=f'Model {Model_VAR_NAME}',
)

ax.scatter(
    obs_df['doy'],
    #obs_df['POC'] * 1000, # for POC to be mg/m3
    obs_df[Obs_VAR_NAME],
    s=22,
    color='tab:orange',
    alpha=0.85,
    label=f'{Obs_VAR_NAME}',
    zorder=3,
)

Var_units = Model_ds.attrs.get('units', '')
ax.set_ylabel(f'{Obs_VAR_NAME} [{Var_units}]')
ax.set_xlabel('Day of Year')
ax.set_title(
    f'Model vs observed {Obs_VAR_NAME} | '
    f'y={Y_SLICE[0]}:{Y_SLICE[1]}, x={X_SLICE[0]}:{X_SLICE[1]}'
)
ax.grid(True, alpha=0.25)
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

print(f'Subdomain used: y={Y_SLICE[0]}:{Y_SLICE[1]}, x={X_SLICE[0]}:{X_SLICE[1]}')
print(f'Number of observation points: {len(obs_df)}')


In [ ]:
# Compare model surface-layer subdomain means with observed surface measurements for:
# Chla (Chl), Phosphate (N1p vs PO4), Nitrate (N3n vs NO3), Ammonium (N4n vs NH4)

import pandas as pd
import matplotlib.pyplot as plt

MODEL_SURFACE_LAYER_INDEX = 10  # top=11, bottom=1 in your convention
USE_DAILY_MEAN = False
ROLLING_WINDOW = None  # e.g. 3 for smoothing, or None

# Model variable -> observation column
VAR_MAP = {
    "Chla": "Chl",
    #"R6c": "POC",
    "ESS": "TSM",
    #"N1p": "PO4",
    "N3n": "NO3",
    "N4n": "NH4",
    "netPPm2": "",
}

# Reuse opened dataset if available; otherwise open yearly files.
try:
    ds
except NameError:
    files = sorted(DATA_DIR.glob(FILE_PATTERN))
    if not files:
        raise FileNotFoundError(f"No files found with pattern: {FILE_PATTERN} in {DATA_DIR}")
    ds = xr.open_mfdataset(
        files,
        combine="nested",
        concat_dim="time",
        decode_times=True,
        data_vars="minimal",
        coords="minimal",
        compat="override",
        join="override",
    )

def _find_time_dim(da: xr.DataArray) -> str:
    for d in da.dims:
        if "time" in d.lower():
            return d
    raise ValueError(f"No time dimension found in {da.dims}")

def _find_vertical_dim(da: xr.DataArray, time_dim: str) -> str | None:
    candidates = ("level", "z", "sigma", "layer", "lev", "depth", "nmesh2_layer_3d")
    for d in da.dims:
        if d != time_dim and any(k in d.lower() for k in candidates):
            return d
    return None

def _drop_duplicate_time(da: xr.DataArray, time_dim: str) -> xr.DataArray:
    time_values = np.asarray(da[time_dim].values)
    _, keep_idx = np.unique(time_values, return_index=True)
    keep_idx = np.sort(keep_idx)
    if keep_idx.size < time_values.size:
        da = da.isel({time_dim: keep_idx})
    return da

def _maybe_smooth(series: xr.DataArray, time_dim: str) -> xr.DataArray:
    out = series
    if USE_DAILY_MEAN:
        out = out.resample({time_dim: "1D"}).mean(skipna=True)
    if ROLLING_WINDOW is not None:
        if ROLLING_WINDOW < 1:
            raise ValueError("ROLLING_WINDOW must be >= 1 or None")
        out = out.rolling({time_dim: ROLLING_WINDOW}, center=True).mean()
    return out

def _model_surface_series(ds: xr.Dataset, model_var: str, layer_index: int) -> tuple[xr.DataArray, str]:
    if model_var not in ds.variables:
        raise KeyError(f"Model variable '{model_var}' not found. Available: {sorted(ds.data_vars)}")

    da = ds[model_var].squeeze(drop=True)
    time_dim = _find_time_dim(da)
    z_dim = _find_vertical_dim(da, time_dim)
    if z_dim is None:
        raise ValueError(f"No vertical dimension found for {model_var}. Dims: {da.dims}")

    xy_dims = [d for d in da.dims if d not in (time_dim, z_dim)]
    if len(xy_dims) != 2:
        raise ValueError(f"Expected 2 horizontal dims for {model_var}, got {xy_dims} from {da.dims}")
    y_dim, x_dim = xy_dims

    y0, y1 = Y_SLICE
    x0, x1 = X_SLICE
    if not (0 <= y0 < y1 <= da.sizes[y_dim]):
        raise IndexError(f"Y_SLICE={Y_SLICE} out of bounds for {y_dim} size {da.sizes[y_dim]}")
    if not (0 <= x0 < x1 <= da.sizes[x_dim]):
        raise IndexError(f"X_SLICE={X_SLICE} out of bounds for {x_dim} size {da.sizes[x_dim]}")

    if layer_index >= da.sizes[z_dim] or layer_index < -da.sizes[z_dim]:
        raise IndexError(f"layer_index={layer_index} out of bounds for '{z_dim}' size {da.sizes[z_dim]}")

    sub = da.isel({y_dim: slice(y0, y1), x_dim: slice(x0, x1)})
    series = sub.isel({z_dim: layer_index}).mean(dim=(y_dim, x_dim), skipna=True)
    series = _drop_duplicate_time(series, time_dim)
    series = _maybe_smooth(series, time_dim)
    return series, time_dim

# Observations: reuse measurement_df if present, otherwise load it.
try:
    measurement_df
except NameError:
    Validation_DATA_DIR = Path("/export/lv9/user/qzhan/archived_output/validation_data/")
    Marsdiep_ts = "pelagic/ts/Jetty_HWseries.csv"
    csv_path = Validation_DATA_DIR / Marsdiep_ts
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV file not found: {csv_path}")
    measurement_df = pd.read_csv(
        csv_path,
        na_values=["NA", ""],
        parse_dates=["timestamp"],
    )

if "timestamp" not in measurement_df.columns:
    raise KeyError("Column 'timestamp' not found in observations CSV.")

obs_df = measurement_df.copy()
obs_df = obs_df.dropna(subset=["timestamp"])
obs_df = obs_df[obs_df["timestamp"].dt.year == 2015].sort_values("timestamp")

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
axes = axes.ravel()

for i, (model_var, obs_col) in enumerate(VAR_MAP.items()):
    ax = axes[i]

    if obs_col not in obs_df.columns:
        ax.text(0.5, 0.5, f"Obs column '{obs_col}' not found", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(f"{model_var} vs {obs_col}")
        ax.set_axis_off()
        continue

    model_series, time_dim = _model_surface_series(ds, model_var, MODEL_SURFACE_LAYER_INDEX)
    model_series = model_series.where(model_series >= 0 if model_var == "Chla" else True)

    obs_var = obs_df[["timestamp", obs_col]].dropna()

    ax.plot(
        model_series[time_dim].values,
        model_series.values,
        lw=2.0,
        color="tab:blue",
        label=f"Model {model_var} (layer {MODEL_SURFACE_LAYER_INDEX})",
    )
    ax.scatter(
        obs_var["timestamp"],
        obs_var[obs_col],
        s=18,
        color="tab:orange",
        alpha=0.85,
        label=f"Obs {obs_col}",
        zorder=3,
    )

    units = ds[model_var].attrs.get("units", "")
    ax.set_ylabel(f"{model_var} / {obs_col} [{units}]" if units else f"{model_var} / {obs_col}")
    ax.set_title(f"{model_var} (model) vs {obs_col} (obs)")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="upper left", fontsize=8)

for ax in axes:
    ax.set_xlabel("Time")

fig.suptitle(
    f"Model vs observed surface concentrations in 2015 | y={Y_SLICE[0]}:{Y_SLICE[1]}, x={X_SLICE[0]}:{X_SLICE[1]}",
    y=1.02,
)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
# Compare model subdomain means with observed measurements for:
# Primary production
if not PP_csv_path.exists():
    raise FileNotFoundError(f'CSV file not found: {PP_csv_path}')

pp_df = pd.read_csv(PP_csv_path, na_values=['NA', ''])
if 'Date' not in pp_df.columns or 'Daily_PP' not in pp_df.columns:
    raise KeyError("CSV must contain columns 'Date' and 'Daily_PP'.")

pp_df['Date'] = pd.to_datetime(pp_df['Date'], errors='coerce')
pp_obs = pp_df[['Date', 'Daily_PP']].dropna(subset=['Date', 'Daily_PP']).copy()
pp_obs = pp_obs.sort_values('Date')
pp_obs = pp_obs[pp_obs['Date'].dt.year == 2015]

if pp_obs.empty:
    raise ValueError('No Daily_PP observations found for 2015 in PP CSV.')

# Reuse opened dataset if available; otherwise open yearly files.
try:
    ds
except NameError:
    files = sorted(DATA_DIR.glob(FILE_PATTERN))
    if not files:
        raise FileNotFoundError(f'No files found with pattern: {FILE_PATTERN} in {DATA_DIR}')
    ds = xr.open_mfdataset(
        files,
        combine='nested',
        concat_dim='time',
        decode_times=True,
        data_vars='minimal',
        coords='minimal',
        compat='override',
        join='override',
    )

MODEL_PP_VAR = 'netPPm2'
if MODEL_PP_VAR not in ds.variables:
    raise KeyError(f"Variable '{MODEL_PP_VAR}' not found. Available: {sorted(ds.data_vars)}")

pp_model_da = ds[MODEL_PP_VAR].squeeze(drop=True)

def _find_time_dim(da: xr.DataArray) -> str:
    for d in da.dims:
        if 'time' in d.lower():
            return d
    raise ValueError(f'No time dimension found in {da.dims}')

def _drop_duplicate_time(da: xr.DataArray, time_dim: str) -> xr.DataArray:
    time_values = np.asarray(da[time_dim].values)
    _, keep_idx = np.unique(time_values, return_index=True)
    keep_idx = np.sort(keep_idx)
    if keep_idx.size < time_values.size:
        da = da.isel({time_dim: keep_idx})
    return da

time_dim = _find_time_dim(pp_model_da)
spatial_dims = [d for d in pp_model_da.dims if d != time_dim]

if len(spatial_dims) >= 2:
    # Use same subdomain as previous cells when 2D horizontal dims exist.
    y_dim, x_dim = spatial_dims[-2], spatial_dims[-1]
    y0, y1 = Y_SLICE
    x0, x1 = X_SLICE
    if 0 <= y0 < y1 <= pp_model_da.sizes[y_dim] and 0 <= x0 < x1 <= pp_model_da.sizes[x_dim]:
        pp_model_da = pp_model_da.isel({y_dim: slice(y0, y1), x_dim: slice(x0, x1)})

spatial_dims = [d for d in pp_model_da.dims if d != time_dim]
if spatial_dims:
    pp_model_series = pp_model_da.mean(dim=spatial_dims, skipna=True)
else:
    pp_model_series = pp_model_da

pp_model_series = _drop_duplicate_time(pp_model_series, time_dim)
# Remove physically invalid negative model PP values
pp_model_series = pp_model_series.where(pp_model_series >= 0)

# Match observation cadence: daily mean from model.
pp_model_daily = pp_model_series.resample({time_dim: '1D'}).mean(skipna=True)
pp_model_df = pd.DataFrame({
    'Date': pd.to_datetime(pp_model_daily[time_dim].values),
    'Model_PP': pp_model_daily.values
})
pp_model_df = pp_model_df.dropna(subset=['Date', 'Model_PP'])
pp_model_df = pp_model_df[pp_model_df['Date'].dt.year == 2015]

if pp_model_df.empty:
    raise ValueError('No model netPPm2 values found for 2015.')

# Plot model and observations.
fig, ax = plt.subplots(figsize=(12, 4.8))

ax.plot(
    pp_model_df['Date'],
    pp_model_df['Model_PP'],
    lw=2.0,
    color='tab:blue',
    label='Model netPPm2 (daily mean)',
)
ax.scatter(
    pp_obs['Date'],
    pp_obs['Daily_PP'],
    s=24,
    color='tab:orange',
    alpha=0.9,
    label='Observed Daily_PP',
    zorder=3,
)

pp_units = ds[MODEL_PP_VAR].attrs.get('units', '')
ax.set_ylabel(f'PP [{pp_units}]' if pp_units else 'PP')
ax.set_xlabel('Time')
ax.set_title(
    f'Model vs observed primary production in 2015 | y={Y_SLICE[0]}:{Y_SLICE[1]}, x={X_SLICE[0]}:{X_SLICE[1]}'
)
ax.grid(True, alpha=0.25)
ax.legend(loc='upper left')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f'PP CSV used: {PP_csv_path}')
print(f'Observation points in 2015: {len(pp_obs)}')
print(f'Model daily points in 2015: {len(pp_model_df)}')

In [ ]:
# ------------------------------ Cell 11: Model vs Copernicus satellite (surface, 2015) ------------------------------
# Template: retrieve Copernicus satellite Chl for the same model subdomain and build a time series.

import os
from pathlib import Path
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import copernicusmarine as cm

import os, getpass
os.environ["COPERNICUSMARINE_SERVICE_USERNAME"] = input("Copernicus username: ")
os.environ["COPERNICUSMARINE_SERVICE_PASSWORD"] = getpass.getpass("Copernicus password: ")

# ---------------- User settings ----------------
SAT_SAVE_DIR = Path("/export/lv9/user/qzhan/archived_output/validation_data//satellite")
SAT_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Example CMEMS Ocean Colour product ID (replace with the one you want from Copernicus catalogue)
# Tip: pick a product with CHL and daily temporal resolution for 2015.
#DATASET_ID = "OCEANCOLOUR_NWS_BGC_HR_L4_NRT_009_209"  # <-- replace if needed
DATASET_ID = "OCEANCOLOUR_NWS_BGC_HR_L3_NRT_009_203"  # <-- replace if needed
SAT_VAR = "CHL"  # <-- adjust if variable name differs (e.g., 'chl')

START_TIME = "2024-01-01T00:00:00"
END_TIME   = "2024-12-31T23:59:59"

# Optional: if needed, set credentials as environment variables beforehand:
# os.environ["COPERNICUSMARINE_SERVICE_USERNAME"] = "your_username"
# os.environ["COPERNICUSMARINE_SERVICE_PASSWORD"] = "your_password"

# ---------------- Reuse/open model dataset ----------------
try:
    ds
except NameError:
    files = sorted(DATA_DIR.glob(FILE_PATTERN))
    if not files:
        raise FileNotFoundError(f"No files found with pattern: {FILE_PATTERN} in {DATA_DIR}")
    ds = xr.open_mfdataset(
        files,
        combine="nested",
        concat_dim="time",
        decode_times=True,
        data_vars="minimal",
        coords="minimal",
        compat="override",
        join="override",
    )

# ---------------- Build geographic bbox from same subdomain ----------------
# Uses model coordinates and the same Y_SLICE/X_SLICE.
# Expecting lon/lat-like variables in ds.
lon_candidates = ("lonc", "lon", "longitude")
lat_candidates = ("latc", "lat", "latitude")

lon_name = next((n for n in lon_candidates if n in ds.variables), None)
lat_name = next((n for n in lat_candidates if n in ds.variables), None)

if lon_name is None or lat_name is None:
    raise KeyError(
        f"Could not find lon/lat variables in dataset. "
        f"Tried lon={lon_candidates}, lat={lat_candidates}."
    )

lon_da = ds[lon_name]
lat_da = ds[lat_name]

# Find horizontal dims from Chla if available; otherwise infer from lon/lat.
if "Chla" in ds.variables:
    chla_tmp = ds["Chla"].squeeze(drop=True)
    dims_lower = [d.lower() for d in chla_tmp.dims]
    time_dim = next((d for d in chla_tmp.dims if "time" in d.lower()), None)
    z_dim = next((d for d in chla_tmp.dims if any(k in d.lower() for k in ("level", "z", "sigma", "layer", "depth", "lev"))), None)
    xy_dims = [d for d in chla_tmp.dims if d not in (time_dim, z_dim)]
    if len(xy_dims) != 2:
        raise ValueError(f"Could not infer 2D horizontal dims from Chla dims: {chla_tmp.dims}")
    y_dim, x_dim = xy_dims
else:
    common_2d = [d for d in lon_da.dims if d in lat_da.dims]
    if len(common_2d) < 2:
        raise ValueError("Could not infer horizontal dims from lon/lat.")
    y_dim, x_dim = common_2d[-2], common_2d[-1]

y0, y1 = Y_SLICE
x0, x1 = X_SLICE

lon_sub = lon_da.isel({y_dim: slice(y0, y1), x_dim: slice(x0, x1)}).values
lat_sub = lat_da.isel({y_dim: slice(y0, y1), x_dim: slice(x0, x1)}).values

lon_min = float(np.nanmin(lon_sub))
lon_max = float(np.nanmax(lon_sub))
lat_min = float(np.nanmin(lat_sub))
lat_max = float(np.nanmax(lat_sub))

# ---------------- Download subset from Copernicus ----------------
# Writes NetCDF to SAT_SAVE_DIR. The file name is determined by the toolbox.
cm.subset(
    dataset_id=DATASET_ID,
    variables=[SAT_VAR],
    minimum_longitude=lon_min,
    maximum_longitude=lon_max,
    minimum_latitude=lat_min,
    maximum_latitude=lat_max,
    start_datetime=START_TIME,
    end_datetime=END_TIME,
    output_directory=str(SAT_SAVE_DIR),
    output_filename="copernicus_satellite_chl_2015.nc",
    force_download=True,
)

sat_path = SAT_SAVE_DIR / "copernicus_satellite_chl_2015.nc"
if not sat_path.exists():
    raise FileNotFoundError(f"Expected satellite file not found: {sat_path}")

sat = xr.open_dataset(sat_path)

if SAT_VAR not in sat.variables:
    raise KeyError(f"Variable '{SAT_VAR}' not found in satellite dataset. Available: {list(sat.variables)}")

sat_da = sat[SAT_VAR].squeeze(drop=True)

# Identify time dim
sat_time_dim = next((d for d in sat_da.dims if "time" in d.lower()), None)
if sat_time_dim is None:
    raise ValueError(f"No time dimension found in satellite variable dims: {sat_da.dims}")

# Spatial average over all non-time dims
sat_spatial_dims = [d for d in sat_da.dims if d != sat_time_dim]
sat_series = sat_da.mean(dim=sat_spatial_dims, skipna=True)

# Optional: remove invalid negative Chl
sat_series = sat_series.where(sat_series >= 0)

# ---------------- Plot satellite time series ----------------
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(
    pd.to_datetime(sat_series[sat_time_dim].values),
    sat_series.values,
    lw=1.8,
    color="tab:green",
    label=f"Satellite {SAT_VAR} (domain mean)",
)
ax.set_title("Satellite Chl (Copernicus) over model subdomain in 2015")
ax.set_xlabel("Time")
ax.set_ylabel(SAT_VAR)
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print("Satellite file:", sat_path)
print(f"Used bbox: lon[{lon_min:.4f}, {lon_max:.4f}], lat[{lat_min:.4f}, {lat_max:.4f}]")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import cmocean
import cartopy.crs as ccrs
import contextily as ctx
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Path to your Excel file
excel_path = "/export/lv9/projects/dws/results/validation/pelagic/SecchiDepth_trawllist241120_IngridTulp.xlsx"

# Load only specific sheet
df = pd.read_excel(excel_path, sheet_name="trawllist_241120")
df["Kd"] = 1.476/df["WATER_VISIBILITY"]+0.3541 # Jacobs et al. (2020) # Obs formula: Kd = 1.476 / Z_Secchi + 0.3541  (Jacobs et al. 2020)
print(df.head())



In [ ]:
# ── Settings ──────────────────────────────────────────────────────────────
MODEL_VAR = "temp"       # light extinction coefficient in the model (m⁻¹)
OBS_VAR   = "temperature"  # observed water temperature column in df_2015
SPINUP_KEY   = "spinup_01"  # which spinup run to use; adjust as needed
Unit = "°C"

v_min = -1 # -100%
v_max = 1 # 100%

In [ ]:
# ── Settings ──────────────────────────────────────────────────────────────
MODEL_VAR = "xEPS"       # light extinction coefficient in the model (m⁻¹)
OBS_VAR   = "Kd"  # observed water temperature column in df_2015
SPINUP_KEY   = "spinup_01"  # which spinup run to use; adjust as needed
Unit = "m⁻¹"

v_min = -3
v_max = 3

In [ ]:
# Compare modelled Temperature (temp) with observed temperature (2015)

import numpy as np
import matplotlib.pyplot as plt
import cmocean
from scipy.spatial import cKDTree

# ── Load model Kd ─────────────────────────────────────────────────────────
ds_model = spinup_datasets.get(SPINUP_KEY, ds)

if MODEL_VAR not in ds_model.variables:
    raise KeyError(
        f"Variable '{MODEL_VAR}' not found in '{SPINUP_KEY}'. "
        f"Available: {sorted(ds_model.data_vars)}"
    )

model_da = ds_model[MODEL_VAR].squeeze(drop=True)
td    = _find_time_dim(model_da)
zd    = _find_vertical_dim(model_da, td)

# Select surface layer if variable is 3D
if zd is not None:
    model_da = model_da.isel({zd: SURFACE_LAYER_INDEX})

model_da = _drop_duplicate_time(model_da, td)

# Infer horizontal dims from model_da (may differ from the global y_dim / x_dim)
h_dims = [d for d in model_da.dims if d != td]
if len(h_dims) != 2:
    raise ValueError(f"Expected 2 horizontal dims after selecting surface, got: {model_da.dims}")
kd_y_dim, kd_x_dim = h_dims

# ── KD-tree: build only from valid (non-NaN) grid points ─────────────────
lon2d = lon_da.values  # (ny, nx)
lat2d = lat_da.values

lon_flat  = lon2d.ravel()
lat_flat  = lat2d.ravel()
valid_mask   = np.isfinite(lon_flat) & np.isfinite(lat_flat)
valid_flat_idx = np.where(valid_mask)[0]   # indices into the full flat array

if valid_flat_idx.size == 0:
    raise ValueError("No valid (non-NaN) coordinates found in lon_da / lat_da.")

tree = cKDTree(np.column_stack([lon_flat[valid_mask], lat_flat[valid_mask]]))

# ── Observations: use df_2015 which already has the Kd column ─────────────
obs = df_2015[["date", "latitude_s", "longitude_s", OBS_VAR]].dropna().copy()
obs["date"] = pd.to_datetime(obs["date"])

model_times = pd.DatetimeIndex(model_da[td].values)

# ── Match each obs to the nearest valid model grid cell + nearest timestep ─
_, idx_in_valid = tree.query(obs[["longitude_s", "latitude_s"]].values)
flat_idx = valid_flat_idx[idx_in_valid]        # map back to full flat grid
iy_arr, ix_arr = np.unravel_index(flat_idx, lon2d.shape)

t_arr = np.array([int(np.argmin(np.abs(model_times - d))) for d in obs["date"]])

obs["var_model"] = [
    float(model_da.isel({kd_y_dim: int(iy), kd_x_dim: int(ix), td: int(it)}).values)
    for iy, ix, it in zip(iy_arr, ix_arr, t_arr)
]
# Remove rows where model Kd is negative (physically invalid) or NaN
obs["var_model"] = obs["var_model"].where(obs["var_model"] > 0)
obs = obs.dropna(subset=[OBS_VAR, "var_model"])
obs["diff"] = (obs["var_model"] - obs[OBS_VAR]) / obs[OBS_VAR]   # positive = model overestimates

print(f"Valid grid points used for KD-tree: {valid_flat_idx.size} / {lon_flat.size}")
print(f"Matched observation points: {len(obs)}")

# ── Scatter plot: obs vs model, coloured by (model − obs) ────────────
vabs = float(np.nanpercentile(np.abs(obs["diff"]), 95))   # robust symmetric limit


fig, ax = plt.subplots(figsize=(7, 6))

sc = ax.scatter(
    obs[OBS_VAR], obs["var_model"],
    c=obs["diff"],
    cmap=cmocean.cm.diff,
    vmin=v_min, vmax=v_max,
    s=45, edgecolors="k", linewidths=0.3, alpha=0.85, zorder=3,
)
cbar = plt.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label(f"Model {OBS_VAR.capitalize()} − Obs {OBS_VAR.capitalize()}  ({Unit})", fontsize=11)

# 1:1 reference line
all_vals = np.concatenate([obs[OBS_VAR].values, obs["var_model"].values])
pad = (all_vals.max() - all_vals.min()) * 0.05
lims = [all_vals.min() - pad, all_vals.max() + pad]
ax.plot(lims, lims, "k--", lw=1.2, label="1:1")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_aspect("equal")

ax.set_xlabel(f"Observed {OBS_VAR.capitalize()}  ({Unit})", fontsize=11)
ax.set_ylabel(f"Modelled {OBS_VAR.capitalize()}  ({Unit})", fontsize=11)
ax.set_title(f"Modelled vs Observed {OBS_VAR.capitalize()} – 2015  ({SPINUP_KEY})", fontsize=12)
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left")

# Summary statistics text box
n    = len(obs)
bias = obs["diff"].mean()
rmse = np.sqrt((obs["diff"] ** 2).mean())
corr = obs[[OBS_VAR, "var_model"]].corr().iloc[0, 1]

ax.text(
    0.05, 0.95,
    f"n = {n}\nBias = {bias:+.3f} \nRMSE = {rmse:.3f} \nR = {corr:.3f}",
    transform=ax.transAxes, va="top", fontsize=10,
    bbox=dict(facecolor="white", alpha=0.75, edgecolor="gray", boxstyle="round,pad=0.3"),
)

plt.tight_layout()
plt.show()


In [ ]:
# Water temperature validation: model vs in-situ measurements (2015)

# water temperature
#VMIN = 10.0
#VMAX = 25.0

# Kd
VMIN = 0
VMAX = 7
# ── Animation: all 2015 measurements ─────────────────────────────────────
df_2015 = (
    df[df["year"] == 2015]
    .dropna(subset=[OBS_VAR, "latitude_s", "longitude_s"])
    .copy()
)
df_2015["date"] = pd.to_datetime(df_2015[["year", "month", "day"]])
dates = sorted(df_2015["date"].unique())
print(f"\nUnique dates with valid water visibility in 2015: {len(dates)}")

fig_a, ax_a = plt.subplots(figsize=(9, 7), subplot_kw={"projection": PROJ})
ax_a.set_extent(MAP_EXTENT, crs=PROJ)
ctx.add_basemap(ax_a, crs=CRS_STR, source=BASEMAP_SRC, zorder=1)
gl_a = ax_a.gridlines(draw_labels=True, linewidth=0.4, color="gray", alpha=0.5)
gl_a.top_labels = False
gl_a.right_labels = False

# Anchor colorbar with a dummy scatter (keeps it fixed across frames)
dummy = ax_a.scatter(
    [], [], c=[], cmap=cmocean.cm.thermal, vmin=VMIN, vmax=VMAX,
    s=50, edgecolors="k", linewidths=0.3,
    transform=ccrs.PlateCarree(), zorder=3,
)
cbar_a = plt.colorbar(dummy, ax=ax_a, pad=0.02, shrink=0.7)
cbar_a.set_label(f"{OBS_VAR.capitalize()} ({Unit})", fontsize=11)
title_a = ax_a.set_title("", fontsize=13)

_scat = [None]  # mutable reference so update() can replace the artist

def _update(frame):
    date = dates[frame]
    df_f = df_2015[df_2015["date"] == date]
    if _scat[0] is not None:
        _scat[0].remove()
    _scat[0] = ax_a.scatter(
        df_f["longitude_s"], df_f["latitude_s"],
        c=df_f[OBS_VAR],
        cmap=cmocean.cm.thermal, vmin=VMIN, vmax=VMAX,
        s=50, edgecolors="k", linewidths=0.3,
        transform=ccrs.PlateCarree(), zorder=3,
    )
    title_a.set_text(
        f"{OBS_VAR.capitalize()} – {date.strftime('%Y-%m-%d')}  [{frame + 1}/{len(dates)}] – 2015  ({SPINUP_KEY})"
    )
    return _scat[0], title_a

anim = FuncAnimation(fig_a, _update, frames=len(dates), interval=500, blit=False)
plt.tight_layout()
HTML(anim.to_jshtml())

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import contextily as ctx
import cmocean

# ── Settings ──────────────────────────────────────────────────────────────
PROJ       = ccrs.PlateCarree()
CRS_STR    = PROJ.to_string()
BASEMAP    = ctx.providers.Esri.WorldShadedRelief
MAP_EXTENT = [3.5, 8.5, 52.5, 55.5]   # NL coastal region

# Use all 2015 matched points
df_all = obs.copy()

# Symmetric colour range for whole-year differences
vabs = float(np.nanpercentile(np.abs(df_all["diff"]), 95))

fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={"projection": PROJ})
ax.set_extent(MAP_EXTENT, crs=PROJ)

# Basemap
ctx.add_basemap(ax, crs=CRS_STR, source=BASEMAP, zorder=1)

# Gridlines
gl = ax.gridlines(draw_labels=True, linewidth=0.4, color="gray", alpha=0.5)
gl.top_labels = False
gl.right_labels = False

# Scatter all points from the entire year
sc = ax.scatter(
    df_all["longitude_s"], df_all["latitude_s"],
    c=df_all["diff"],
    cmap=cmocean.cm.diff,
    vmin=v_min, vmax=v_max,
    s=55, edgecolors="k", linewidths=0.3,
    transform=ccrs.PlateCarree(), zorder=3,
)

cbar = plt.colorbar(sc, ax=ax, pad=0.02, shrink=0.7)
cbar.set_label(f"Model − Obs {OBS_VAR.capitalize()}  ({Unit})", fontsize=11)

ax.set_title(f"Model–Obs {OBS_VAR}  Difference (All 2015 Observations) ({SPINUP_KEY})", fontsize=14)
plt.tight_layout()
plt.show()
